# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:
# Run once if packages are missing
#!pip install -r requirements.txt

In [1]:
# ==============================
# 1. Imports
# ==============================

import os
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import AzureOpenAI
from dotenv import load_dotenv

In [2]:
# ==============================
# 2. Load environment variables
# ==============================

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# ==============================
# 3. Load knowledge base
# ==============================

with open("./data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


Chunks:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [4]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 4


In [5]:
# ==============================
# 6. User question
# ==============================

#question = "Which ocean is the largest on Earth?"
question = "Is Great Wall of China visible from the Moon?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

# You can print out the embedding just in case you want to see how it looks
#print("Question Embedding:", question_embedding)



Question: Is Great Wall of China visible from the Moon?


In [6]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


Retrieved context:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.


In [7]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information awailable" if there is no answer in the context

# Context:
{context}

# Question:
{question}

# Answer:
"""

#print("Prompt:", prompt)


In [9]:


# ==============================
# 10. Call GPT-4o to generate the answer
# ==============================

response = client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)


LLM Answer:
Yes, the Great Wall of China is visible from the Moon.
